# Week 5: IDS and one-page Nôm recognition preparation

This notebook uses the pinned course dictionaries. It demonstrates **text-level** IDS tokenization; it does not segment radical regions from pixels. Run the resource download command in the handout before starting.

## 1. IDS as a prefix expression

An IDC comes before its operands: `⿰日月` means 日 is left of 月. Binary IDCs such as `⿰`, `⿱`, `⿴`, and `⿻` take two operands; `⿲` and `⿳` take three. A direct CJKVI IDS describes one layer. The recognizer's expanded IDS recursively replaces components until it reaches its terminal vocabulary.

In [ ]:
import json
import os
from urllib.parse import quote
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

import nom_ocr_lab
from nom_ocr_lab.annotations import read_line_annotations
from nom_ocr_lab.recognition import (
    load_expanded_ids,
    load_vocabulary,
    reverse_ids,
    text_to_ids,
)

workspace_value = os.environ.get("LAB_WORKSPACE")
if not workspace_value:
    raise RuntimeError("Set LAB_WORKSPACE before starting Jupyter")
workspace = Path(workspace_value)
course_root = Path(nom_ocr_lab.__file__).resolve().parents[2]
ids_root = course_root / "nom-ids"
if not ids_root.is_dir():
    raise RuntimeError(f"Bundled IDS resources are missing: {ids_root}")
nomnaocr_root = Path(os.environ.get("NOMNAOCR_ROOT", "/data/shared/project2026/nomnaocr"))
image_root = nomnaocr_root / "images"
line_label_root = nomnaocr_root / "line_labels"
print(f"Lab workspace: {workspace}")
print(f"Bundled IDS resources: {ids_root}")
print(f"Read-only NomNaOCR corpus: {nomnaocr_root}")

In [ ]:
expanded = load_expanded_ids(ids_root / "ids_exp.txt")
vocabulary = load_vocabulary(ids_root / "vocab_ids.txt")
ids_to_characters = reverse_ids(expanded)
print(f"Expanded character entries: {len(expanded):,}")
print(f"Recognizer tokens, in checkpoint order: {len(vocabulary):,}")

## 2. Runnable reference converter and exact Chinese check

`text_to_ids` emits one structured record per non-whitespace Unicode code point. Copying the whole string into one field would lose the character boundaries needed later. Reuse this provided converter in your team repository; each record also includes a direct `zi_tools_url`.

In [ ]:
expected = {
    "休": "⿰亻木",
    "好": "⿰女子",
    "字": "⿱宀子",
    "明": "⿰日月",
    "林": "⿰木木",
}
reference = text_to_ids("休好字明林", expanded)
observed = {record.character: record.ids for record in reference}
assert observed == expected, (observed, expected)
for record in reference:
    candidates = ids_to_characters[record.ids]
    zi_tools_url = f"https://zi.tools/zi/{quote(record.character)}"
    print(record.codepoint, record.character, record.ids, list(record.tokens), candidates, zi_tools_url)
print("Five Chinese reference conversions passed.")

Before continuing, write `明`, `休`, and one nested character as IDS by hand. Compare the spatial relation you chose with the direct column printed above, then explain why a direct and an expanded IDS can differ.

## 3. Audit the full NomNaOCR corpus

The next two cells discover every page in the full read-only NomNaOCR corpus under `/data/shared/project2026/nomnaocr/images`, with matching line transcriptions under `line_labels`, then report IDS coverage and OOV examples. Set `NOMNAOCR_ROOT` only when you need to point at an equivalent read-only corpus. The Weeks 3–4 YOLO archive contains character-box annotations for detector evaluation, not line transcriptions, and cannot be used for this IDS audit.

In [ ]:
image_suffixes = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
available = sorted(
    path.relative_to(image_root)
    for path in image_root.rglob("*")
    if path.is_file() and path.suffix.lower() in image_suffixes
)
if not available:
    raise RuntimeError(f"No NomNaOCR page images found below {image_root}")
print(f"Corpus pages available: {len(available)}")

In [ ]:
corpus_records = []
oov_examples = {}
for relative_path in available:
    corpus_line_path = line_label_root / relative_path.with_suffix(".txt")
    if not corpus_line_path.is_file():
        raise RuntimeError(
            f"Missing line transcription for {relative_path}: {corpus_line_path}. "
            "The full NomNaOCR corpus must contain matching line-level transcriptions."
        )
    page_records = [
        record
        for annotation in read_line_annotations(corpus_line_path)
        for record in text_to_ids(annotation.text, expanded)
    ]
    corpus_records.extend(page_records)
    page_oov_counts = Counter(record.character for record in page_records if record.status == "oov")
    for character in page_oov_counts:
        oov_examples.setdefault(character, (relative_path, corpus_line_path))

covered_total = sum(record.status == "covered" for record in corpus_records)
character_total = len(corpus_records)
coverage = covered_total / character_total if character_total else 0.0
oov_counts = Counter(record.character for record in corpus_records if record.status == "oov")
print(f"Corpus pages scanned: {len(available)}")
print(f"Corpus IDS coverage: {covered_total}/{character_total} = {coverage:.2%}")
print(
    f"Missing IDS: {sum(oov_counts.values())} occurrences of "
    f"{len(oov_counts)} distinct characters"
)
print("Missing IDS examples (character | Unicode | occurrences | image path | label path)")
for character, occurrences in oov_counts.most_common(10):
    relative_path, label_path = oov_examples[character]
    print(
        f"{character} | U+{ord(character):04X} | {occurrences} | "
        f"{image_root / relative_path} | {label_path}"
    )
if not oov_counts:
    print("No missing IDS characters in this durable corpus.")

## 4. Audit one page: image, line boxes, and transcriptions

After reviewing the corpus-wide results, set `WEEK05_PAGE` to a path relative to `images` to choose a page; otherwise the first sorted page is used. This cell loads the page image and its matching line-label file, overlays every line polygon, and prints each transcription. The numbered polygons and numbered transcription list refer to the same line annotation.

In [ ]:
requested = os.environ.get("WEEK05_PAGE")
relative_image = Path(requested) if requested else available[0]
if relative_image not in available:
    raise RuntimeError(f"WEEK05_PAGE is not in the NomNaOCR corpus: {relative_image}")

image_path = image_root / relative_image
line_path = line_label_root / relative_image.with_suffix(".txt")
if not line_path.is_file():
    raise RuntimeError(f"Missing line transcription for {relative_image}: {line_path}")
annotations = read_line_annotations(line_path)
with Image.open(image_path) as source_image:
    page_image = source_image.convert("RGB")

print(f"Selected page: {relative_image}")
print(f"Image: {image_path} ({page_image.width} × {page_image.height} pixels)")
print(f"Line label file: {line_path}")
print(f"Line transcriptions: {len(annotations)}")

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(page_image)
for line_number, annotation in enumerate(annotations, start=1):
    polygon = plt.Polygon(annotation.points, fill=False, edgecolor="lime", linewidth=1.2)
    ax.add_patch(polygon)
    x, y = annotation.points[0]
    ax.text(x, y, str(line_number), color="black", backgroundcolor="lime", fontsize=8)
ax.set_title(f"Line annotations: {relative_image}")
ax.axis("off")
plt.show()

for line_number, annotation in enumerate(annotations, start=1):
    print(f"{line_number:>3}: {annotation.text}")

## Submission reflection

1. What information does `⿰` add that an unordered radical set would lose?
2. Why must `vocab_ids.txt` retain its original order?
3. Which page characters were OOV, and were they uncertain marks, punctuation, or ideographs?
4. What direct/expanded or glyph-region differences appeared in zi.tools?
5. Why would pairing the first detected box with the first transcription character create false ground truth?